# Notebook 01 — Baseline DQN on CartPole-v1 (v3)

APBIO project, 2025/26.

Third pass at the baseline. In v2, 3 seeds all looked like they topped out
at ~50 return with exactly 100 logged episodes each. That number is suspicious:
SB3's `ep_info_buffer` is a deque of capacity 100 by default, and my previous
`EpisodeLogger` read from that buffer. Once it filled up, new episodes
replaced older ones silently and my `self._seen` counter lost track — so I
was only recording the first 100 episodes and discarding the rest.

The fix: wrap the env in `Monitor`, then read `info['episode']` from the step
info dict. Every finished episode shows up exactly once, no buffer overflow.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, pathlib

PROJECT_ROOT = pathlib.Path('/content/drive/MyDrive/apbio_project')
(PROJECT_ROOT / 'logs').mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / 'figures').mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)
print('cwd:', os.getcwd())

## 2. Install packages

In [ ]:
%pip install -q 'gymnasium>=0.29' 'stable-baselines3>=2.3'

In [ ]:
import gymnasium as gym
import stable_baselines3 as sb3
import torch, numpy as np

print('gymnasium:', gym.__version__)
print('sb3      :', sb3.__version__)
print('torch    :', torch.__version__)

## 3. Seed helper

In [ ]:
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## 4. Fixed per-episode CSV logger

This version reads from `info['episode']`, which the `Monitor` wrapper sets
at the end of every episode. Unlike the old buffer-based version, no
finished episode is ever missed, no matter how many the agent plays.

In [ ]:
import csv
from pathlib import Path
from stable_baselines3.common.callbacks import BaseCallback

class EpisodeLogger(BaseCallback):
    def __init__(self, csv_path):
        super().__init__()
        self.csv_path = Path(csv_path)
        self.csv_path.parent.mkdir(parents=True, exist_ok=True)
        self._ep = 0
        with self.csv_path.open('w', newline='') as f:
            csv.writer(f).writerow(['episode', 'timestep', 'return', 'length'])

    def _on_step(self):
        # Monitor wrapper puts an 'episode' dict in info when an episode ends.
        infos = self.locals.get('infos', [])
        finished = []
        for info in infos:
            if 'episode' in info:
                self._ep += 1
                finished.append([
                    self._ep,
                    self.num_timesteps,
                    float(info['episode']['r']),
                    int(info['episode']['l']),
                ])
        if finished:
            with self.csv_path.open('a', newline='') as f:
                csv.writer(f).writerows(finished)
        return True

## 5. Training function

The only change from before is the `Monitor(env)` wrap. Hyperparameters
are the same RL Zoo defaults.

In [ ]:
import time
from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor

TOTAL_TIMESTEPS = 50_000

def train_baseline(seed, total_timesteps=TOTAL_TIMESTEPS, verbose=1):
    set_seed(seed)
    env = Monitor(gym.make('CartPole-v1'))
    log_path = PROJECT_ROOT / 'logs' / f'baseline_seed{seed}.csv'

    model = DQN(
        policy='MlpPolicy',
        env=env,
        learning_rate=2.3e-3,
        buffer_size=100_000,
        learning_starts=1_000,
        batch_size=64,
        tau=1.0,
        gamma=0.99,
        train_freq=256,
        gradient_steps=128,
        target_update_interval=10,
        exploration_fraction=0.16,
        exploration_final_eps=0.04,
        policy_kwargs=dict(net_arch=[256, 256]),
        seed=seed,
        verbose=verbose,
    )

    cb = EpisodeLogger(csv_path=str(log_path))
    t0 = time.time()
    model.learn(total_timesteps=total_timesteps, callback=cb, log_interval=10)
    elapsed = time.time() - t0

    return log_path, elapsed

## 6. Run 3 seeds

Same 3 seeds as before. With the logging fix, I expect to see many more
than 100 episodes per run, and the later ones should reach high returns.

In [ ]:
SEEDS = [0, 42, 123]
run_log = {}

for s in SEEDS:
    print(f'\n=== seed {s} ===')
    log_path, elapsed = train_baseline(s)
    run_log[s] = log_path
    print(f'seed {s} finished in {elapsed:.1f}s, log -> {log_path}')

## 7. Compare the three seeds

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))

for s in SEEDS:
    df = pd.read_csv(run_log[s])
    smooth = df['return'].rolling(20, min_periods=1).mean()
    ax.plot(df['episode'], smooth, label=f'seed {s}')

ax.axhline(500, color='grey', linestyle='--', linewidth=1, label='max (500)')
ax.set_xlabel('Episode')
ax.set_ylabel('Return (20-ep rolling mean)')
ax.set_title('Baseline DQN on CartPole-v1 — 3 seeds')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
fig.tight_layout()

out = PROJECT_ROOT / 'figures' / 'baseline_3seeds.png'
fig.savefig(out, dpi=150)
plt.show()
print('figure:', out)

## 8. Per-seed summary

In [ ]:
rows = []
for s in SEEDS:
    df = pd.read_csv(run_log[s])
    rows.append({
        'seed': s,
        'episodes': len(df),
        'first_10_mean': round(df['return'].head(10).mean(), 1),
        'last_10_mean': round(df['return'].tail(10).mean(), 1),
        'max_return': int(df['return'].max()),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

## What to look for now

- Each run should log hundreds or thousands of episodes (not exactly 100).
- The smoothed curves should clearly rise toward 500.
- `last_10_mean > 400` on most seeds = baseline works, move on to notebook 02.